# Machine Learning aplicado a procesos empresariales -- Clase 3
## Clasificacion binaria y metricas de evaluacion

**Objetivo de la sesion:**
1. Entender que es un problema de clasificacion binaria.
2. Entrenar un modelo de Regresion Logistica.
3. Evaluar el modelo con la matriz de confusion y metricas (Accuracy, Precision, Recall).
4. Reflexionar sobre el costo del error en decisiones empresariales.

## 1. Regresion vs Clasificacion: cual es la diferencia?

En la clase anterior trabajamos **regresion**: predecir un numero (ventas en dolares).

Hoy trabajamos **clasificacion binaria**: predecir una de dos categorias.

| Aspecto | Regresion | Clasificacion binaria |
|---------|-----------|----------------------|
| Variable objetivo | Numerica (ej. $383.72) | Categorica con 2 opciones (ej. 0 o 1) |
| Pregunta tipica | Cuanto? | Si o No? |
| Ejemplo de negocio | Cuanto vendera esta tienda? | Este cliente se ira? |
| Modelo tipico | Regresion Lineal | Regresion Logistica |
| Metricas | MAE, RMSE, R2 | Accuracy, Precision, Recall |

**Ejemplos de clasificacion binaria en negocios:**
- Este cliente abandonara el servicio? (SI / NO)
- Esta transaccion es fraude? (SI / NO)
- Este correo es spam? (SI / NO)
- El paciente tiene la enfermedad? (SI / NO)

## 2. El problema de hoy: abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere **predecir cuales clientes
van a abandonar el servicio** (churn) para poder tomar acciones preventivas.

**Por que es importante?**
- Conseguir un cliente nuevo cuesta entre 5x y 25x mas que retener uno existente.
- Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.

**Variables del dataset:**

| Variable | Descripcion | Tipo |
|----------|-------------|------|
| antiguedad_meses | Meses como cliente (1 a 72) | Numerica |
| factura_mensual | Valor mensual del plan en dolares (20 a 120) | Numerica |
| llamadas_soporte | Llamadas al soporte tecnico (0 a 10) | Numerica |
| contrato | Tipo de contrato: Mensual, Anual, Dos_anios | Categorica |
| satisfaccion | Nivel de satisfaccion (1 a 5) | Numerica |
| **abandono** | **1 = abandono, 0 = permanece (VARIABLE OBJETIVO)** | **Binaria** |

## 3. Preparacion del entorno

Importamos todas las librerias que necesitamos. Cada una tiene un rol especifico
que se explica en los comentarios.

In [ ]:
# --- Librerias para manejo de datos ---
import pandas as pd          # Tablas de datos (DataFrames)
import numpy as np            # Operaciones matematicas

# --- Librerias de Machine Learning (scikit-learn) ---
from sklearn.model_selection import train_test_split   # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer          # Aplicar transformaciones por tipo de columna
from sklearn.preprocessing import OneHotEncoder        # Convertir categorias a numeros (0/1)
from sklearn.pipeline import Pipeline                  # Encadenar preprocesamiento + modelo
from sklearn.linear_model import LogisticRegression    # Modelo de clasificacion

# --- Metricas de evaluacion para clasificacion ---
from sklearn.metrics import (
    confusion_matrix,          # Matriz de confusion (tabla de aciertos y errores)
    classification_report,     # Reporte con Precision, Recall, F1-Score
    accuracy_score,            # Porcentaje de predicciones correctas
)

# --- Visualizacion ---
import matplotlib.pyplot as plt  # Graficos

print("Librerias cargadas correctamente.")

## 4. Cargar y explorar el dataset

Siempre antes de construir un modelo debemos entender nuestros datos:
- Cuantos registros hay?
- Que tipo de datos tiene cada columna?
- Hay valores faltantes?
- Como esta distribuida la variable objetivo?

In [ ]:
# Cargar el archivo CSV en un DataFrame (tabla de datos)
df = pd.read_csv("abandono_clientes.csv")

# Mostrar las primeras 5 filas para verificar que se cargo correctamente
print("Primeras 5 filas del dataset:")
df.head()

In [ ]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

In [ ]:
# Estadisticas descriptivas de las variables numericas
df.describe().round(2)

In [ ]:
# ---------------------------------------------------------------
# DISTRIBUCION DE LA VARIABLE OBJETIVO
# ---------------------------------------------------------------
# Esto es MUY importante en clasificacion.
# Si el 99% de clientes no abandona, un modelo que siempre diga "no abandona"
# tendria 99% de accuracy pero seria completamente inutil.
# Necesitamos ver que tan balanceadas estan las clases.

conteo = df["abandono"].value_counts()
porcentaje = df["abandono"].value_counts(normalize=True) * 100

print("Distribucion de la variable objetivo (abandono):")
print(f"  Permanece (0): {conteo[0]} clientes ({porcentaje[0]:.1f}%)")
print(f"  Abandona  (1): {conteo[1]} clientes ({porcentaje[1]:.1f}%)")
print()
print("Las clases estan moderadamente desbalanceadas.")
print("Hay mas clientes que permanecen que clientes que abandonan.")
print("Esto es lo tipico en la realidad: la mayoria de clientes NO se van.")

## 5. Analisis exploratorio: como se comportan las variables segun abandono?

Antes de entrenar el modelo, veamos si las variables tienen relacion con el abandono.
Esto nos ayuda a entender que factores podrian influir en la decision del cliente.

In [ ]:
# Comparar el promedio de cada variable numerica segun si el cliente abandono o no
# groupby("abandono") separa los datos en dos grupos: 0 (permanece) y 1 (abandona)
# .mean() calcula el promedio de cada variable en cada grupo

comparacion = df.groupby("abandono")[["antiguedad_meses", "factura_mensual",
                                       "llamadas_soporte", "satisfaccion"]].mean().round(2)

# Renombrar el indice para que sea mas legible
comparacion.index = ["Permanece (0)", "Abandona (1)"]

print("Promedio de cada variable segun abandono:")
print()
comparacion

In [ ]:
# Analisis del tipo de contrato vs abandono
# pd.crosstab crea una tabla cruzada: filas = contrato, columnas = abandono
# normalize="index" muestra porcentajes por fila (cada tipo de contrato suma 100%)

tabla_contrato = pd.crosstab(
    df["contrato"],
    df["abandono"],
    normalize="index"  # Porcentajes por fila
).round(3) * 100

tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]

print("Porcentaje de abandono por tipo de contrato:")
print()
tabla_contrato

**Lectura del analisis exploratorio:**

Si observas las tablas anteriores, los clientes que abandonan tienden a:
- Tener **menos antiguedad** (son mas nuevos).
- Hacer **mas llamadas a soporte** (estan insatisfechos).
- Tener **menor nivel de satisfaccion**.
- Tener contratos **mensuales** (sin compromiso a largo plazo).

Estas observaciones tienen sentido desde el punto de vista del negocio.
Ahora vamos a ver si el modelo puede aprender estos patrones automaticamente.

## 6. Preparar los datos para el modelo

Los pasos son los mismos que en la clase anterior:
1. Separar variables predictoras (X) de la variable objetivo (y).
2. Dividir en datos de entrenamiento (80%) y prueba (20%).

**Importante:** El modelo aprende SOLO con los datos de entrenamiento.
Los datos de prueba se usan SOLO para evaluar. Si evaluamos con los mismos
datos con los que entrenamos, el modelo podria estar "memorizando" en vez de
aprendiendo, y no sabriamos si funciona con datos nuevos.

In [ ]:
# --- Paso 1: Separar X (variables predictoras) e y (variable objetivo) ---
X = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte",
        "contrato", "satisfaccion"]]
y = df["abandono"]

# --- Paso 2: Dividir en entrenamiento (80%) y prueba (20%) ---
# stratify=y mantiene la misma proporcion de abandono en ambos conjuntos.
# Ejemplo: si hay 35% de abandono en el total, habra ~35% en entrenamiento y ~35% en prueba.
# Esto es importante cuando las clases estan desbalanceadas.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% para prueba
    random_state=42,     # Semilla para reproducibilidad
    stratify=y           # Mantener proporcion de clases
)

print(f"Datos de entrenamiento: {X_train.shape[0]} clientes")
print(f"Datos de prueba:        {X_test.shape[0]} clientes")
print()
print(f"Proporcion de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporcion de abandono en prueba:        {y_test.mean()*100:.1f}%")
print()
print("Las proporciones son similares gracias al parametro stratify=y.")

## 7. Construir el pipeline y entrenar el modelo

### Que es la Regresion Logistica?

A pesar de su nombre, la Regresion Logistica es un modelo de **clasificacion**, no de regresion.

Funciona asi:
1. Calcula una puntuacion lineal (igual que la regresion lineal).
2. Pasa esa puntuacion por una **funcion sigmoide** que la convierte en una probabilidad entre 0 y 1.
3. Si la probabilidad es mayor a 0.5, predice la clase 1 (abandona); si no, predice 0 (permanece).

Es el modelo mas simple de clasificacion y un excelente punto de partida.

In [ ]:
# --- Definir columnas por tipo ---
numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]

# --- Preprocesamiento ---
# Las columnas numericas se dejan tal cual (passthrough).
# La columna categorica (contrato) se convierte a columnas binarias con OneHotEncoder.
# Ejemplo: contrato="Mensual" -> contrato_Mensual=1, contrato_Anual=0, contrato_Dos_anios=0
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# --- Pipeline: preprocesamiento + modelo ---
# El pipeline garantiza que los datos siempre se preprocesen antes de llegar al modelo.
# max_iter=1000 le da al modelo suficientes iteraciones para converger (encontrar la solucion).
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# --- Entrenar el modelo ---
# .fit() hace que el modelo aprenda los patrones de los datos de entrenamiento.
# Internamente calcula los pesos (coeficientes) que mejor separan a los clientes
# que abandonan de los que permanecen.
pipe.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

## 8. Hacer predicciones

Ahora usamos el modelo entrenado para predecir el abandono en los datos de prueba
(datos que el modelo **nunca vio** durante el entrenamiento).

In [ ]:
# Generar predicciones con los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones vs valores reales
comparacion_pred = pd.DataFrame({
    "Real":      y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto":  ["Si" if r == p else "NO" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs valores reales:")
print()
comparacion_pred

## 9. Evaluacion del modelo: Accuracy (exactitud)

La metrica mas basica es el **Accuracy**: que porcentaje de predicciones fueron correctas.

```
Accuracy = Predicciones correctas / Total de predicciones
```

**CUIDADO:** Accuracy puede ser enganosa cuando las clases estan desbalanceadas.

Ejemplo: si el 95% de clientes NO abandona, un modelo que siempre diga "no abandona"
tendria 95% de accuracy, pero jamas detectaria a un cliente en riesgo. Seria inutil.

Por eso necesitamos metricas adicionales.

In [ ]:
# Calcular el accuracy
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy (exactitud): {acc:.4f} ({acc*100:.1f}%)")
print()
print(f"Esto significa que el modelo acerto {int(acc * len(y_test))} de {len(y_test)} predicciones.")
print()
print("Pero... cuantos clientes que realmente abandonaron logro detectar?")
print("Para responder esa pregunta, necesitamos la MATRIZ DE CONFUSION.")

## 10. Matriz de Confusion

La matriz de confusion es una tabla 2x2 que desglosa las predicciones en 4 categorias:

```
                        PREDICCION DEL MODELO
                    |  Permanece (0)  |  Abandona (1)  |
  ---------------------------------------------------------
  REALIDAD  Perm(0) |       VN        |       FP       |
            Aban(1) |       FN        |       VP       |
  ---------------------------------------------------------
```

Donde:
- **VP (Verdadero Positivo):** El modelo dijo "abandona" y SI abandono. ACIERTO.
- **VN (Verdadero Negativo):** El modelo dijo "permanece" y SI permanecio. ACIERTO.
- **FP (Falso Positivo):** El modelo dijo "abandona" pero NO abandono. ERROR (falsa alarma).
- **FN (Falso Negativo):** El modelo dijo "permanece" pero SI abandono. ERROR (no lo detecto).

**VP y VN son los aciertos. FP y FN son los errores.**

In [ ]:
# Calcular la matriz de confusion
cm = confusion_matrix(y_test, y_pred)

# Extraer los 4 valores de la matriz
VN, FP, FN, VP = cm.ravel()

print("=" * 50)
print("          MATRIZ DE CONFUSION")
print("=" * 50)
print()
print(f"                  Prediccion del modelo")
print(f"                  Permanece(0)  Abandona(1)")
print(f"  Real Perm.(0)       {VN:3d}           {FP:3d}")
print(f"  Real Aban.(1)       {FN:3d}           {VP:3d}")
print()
print("=" * 50)
print()
print("Lectura de la matriz:")
print(f"  Verdaderos Negativos (VN): {VN} -- Dijo 'permanece' y acerto")
print(f"  Falsos Positivos     (FP): {FP} -- Dijo 'abandona' pero no se fue (falsa alarma)")
print(f"  Falsos Negativos     (FN): {FN} -- Dijo 'permanece' pero SI se fue (NO lo detecto)")
print(f"  Verdaderos Positivos (VP): {VP} -- Dijo 'abandona' y acerto")

In [ ]:
# --- Visualizacion grafica de la matriz de confusion ---
fig, ax = plt.subplots(figsize=(6, 5))

# Dibujar la matriz como un mapa de calor
im = ax.imshow(cm, cmap="Blues")

# Etiquetas de los ejes
labels = ["Permanece (0)", "Abandona (1)"]
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel("Prediccion del modelo")
ax.set_ylabel("Valor real")
ax.set_title("Matriz de Confusion")

# Colocar los numeros dentro de cada celda
nombres = [[f"VN\n{VN}", f"FP\n{FP}"],
           [f"FN\n{FN}", f"VP\n{VP}"]]
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, nombres[i][j], ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 11. Precision, Recall y F1-Score

Ahora que entendemos la matriz de confusion, podemos calcular metricas mas informativas:

### Precision (de la clase "abandona")
De todos los clientes que el modelo dijo que abandonarian, cuantos REALMENTE abandonaron?

```
Precision = VP / (VP + FP)
```

**Interpretacion:** Si la Precision es 80%, significa que de cada 10 clientes que el modelo
marca como "va a abandonar", 8 realmente se van y 2 son falsas alarmas.

---

### Recall (sensibilidad, de la clase "abandona")
De todos los clientes que REALMENTE abandonaron, cuantos logro detectar el modelo?

```
Recall = VP / (VP + FN)
```

**Interpretacion:** Si el Recall es 60%, significa que de cada 10 clientes que realmente
se fueron, el modelo solo detecto 6 y dejo pasar 4 sin aviso.

---

### F1-Score
Es el promedio armonico entre Precision y Recall. Util cuando quieres un solo numero
que resuma ambas metricas.

```
F1 = 2 * (Precision * Recall) / (Precision + Recall)
```

In [ ]:
# --- Calcular Precision y Recall manualmente para entender la formula ---
precision_manual = VP / (VP + FP) if (VP + FP) > 0 else 0
recall_manual = VP / (VP + FN) if (VP + FN) > 0 else 0
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0

print("CALCULO MANUAL (para la clase 'abandona'):")
print(f"  Precision = VP / (VP + FP) = {VP} / ({VP} + {FP}) = {precision_manual:.4f} ({precision_manual*100:.1f}%)")
print(f"  Recall    = VP / (VP + FN) = {VP} / ({VP} + {FN}) = {recall_manual:.4f} ({recall_manual*100:.1f}%)")
print(f"  F1-Score  = 2 * (P * R) / (P + R) = {f1_manual:.4f} ({f1_manual*100:.1f}%)")
print()
print("Ahora veamos el reporte completo generado por scikit-learn:")

In [ ]:
# --- Reporte completo de clasificacion ---
# classification_report genera un reporte con Precision, Recall y F1
# para AMBAS clases (0 y 1) y promedios generales.
print("=" * 55)
print("       REPORTE DE CLASIFICACION")
print("=" * 55)
print()
print(classification_report(
    y_test, y_pred,
    target_names=["Permanece (0)", "Abandona (1)"]
))
print("=" * 55)
print()
print("COMO LEER ESTE REPORTE:")
print("  - Cada fila es una clase (Permanece o Abandona).")
print("  - precision: de los que el modelo dijo que son de esta clase, cuantos acerto.")
print("  - recall: de los que realmente son de esta clase, cuantos detecto.")
print("  - f1-score: promedio armonico entre precision y recall.")
print("  - support: cantidad de muestras reales de cada clase en el set de prueba.")

## 12. El costo del error: por que Recall importa mas aqui

En problemas de negocio, **no todos los errores cuestan lo mismo.**

En nuestro caso de abandono de clientes, hay dos tipos de errores:

### Error tipo 1: Falso Positivo (FP)
El modelo dice "este cliente va a abandonar" pero el cliente se queda.

**Consecuencia:** Le ofrecemos un descuento o llamamos a un cliente que no lo necesitaba.
**Costo:** Bajo. Gastamos un poco de tiempo/dinero en retencion innecesaria, pero no perdemos al cliente.

### Error tipo 2: Falso Negativo (FN)
El modelo dice "este cliente se va a quedar" pero el cliente SE VA.

**Consecuencia:** No hacemos nada y perdemos al cliente.
**Costo:** Alto. Perdemos los ingresos futuros de ese cliente, mas el costo de adquirir uno nuevo.

---

### Conclusion

```
Que es peor?

  (a) Contactar a un cliente que NO se iba a ir (Falso Positivo)
      -> Costo: una llamada o descuento innecesario

  (b) NO detectar a un cliente que SI se va (Falso Negativo)
      -> Costo: perder al cliente y sus ingresos futuros
```

**Respuesta: (b) es mucho peor.**

Por eso, en este problema nos importa mas el **Recall** (detectar la mayor cantidad
posible de clientes que realmente abandonan), incluso si eso genera algunas falsas alarmas.

En otros problemas puede ser diferente. Por ejemplo, en un filtro de spam,
un Falso Positivo (marcar un correo importante como spam) puede ser peor que
un Falso Negativo (dejar pasar un spam al inbox).

In [ ]:
# --- Simulacion del costo economico de los errores ---
# Supongamos estos costos estimados:
costo_fp = 20     # Costo de contactar a un cliente que no se iba ($20: llamada + descuento)
costo_fn = 500    # Costo de perder a un cliente ($500: ingresos perdidos)

# Calcular el costo total de los errores del modelo
costo_total_fp = FP * costo_fp
costo_total_fn = FN * costo_fn
costo_total = costo_total_fp + costo_total_fn

# Calcular el costo si NO usaramos modelo (todos los que abandonan se pierden)
total_abandonos_reales = VP + FN  # Total de clientes que realmente abandonaron
costo_sin_modelo = total_abandonos_reales * costo_fn

# Ahorro gracias al modelo
ahorro = costo_sin_modelo - costo_total

print("=" * 55)
print("    ANALISIS DE COSTO DE LOS ERRORES")
print("=" * 55)
print()
print(f"  Costo por Falso Positivo (falsa alarma): ${costo_fp}")
print(f"  Costo por Falso Negativo (no detectado): ${costo_fn}")
print()
print(f"  Falsos Positivos: {FP} x ${costo_fp} = ${costo_total_fp:,}")
print(f"  Falsos Negativos: {FN} x ${costo_fn} = ${costo_total_fn:,}")
print(f"  Costo total de errores: ${costo_total:,}")
print()
print(f"  Si NO usaramos modelo (perder a todos): ${costo_sin_modelo:,}")
print(f"  Ahorro gracias al modelo: ${ahorro:,}")
print()
print("  Los Falsos Negativos (clientes no detectados) representan")
print(f"  el {costo_total_fn/costo_total*100:.0f}% del costo total de errores.")
print("  Esto confirma que en este problema, el Recall es critico.")

## 13. Resumen de la clase

### Que aprendimos hoy?

1. **Clasificacion binaria:** predecir SI o NO (abandona o permanece).

2. **Regresion Logistica:** modelo que calcula la probabilidad de pertenecer a una clase.

3. **Matriz de confusion:** tabla que desglosa aciertos y errores en VP, VN, FP, FN.

4. **Metricas:**
   - **Accuracy** = predicciones correctas / total. Simple pero puede ser enganosa.
   - **Precision** = de los que dije positivos, cuantos realmente lo eran.
   - **Recall** = de los positivos reales, cuantos detecte.
   - **F1-Score** = promedio armonico entre Precision y Recall.

5. **Costo del error:** en cada problema de negocio, un tipo de error puede ser
   mucho mas costoso que el otro. Esto determina cual metrica priorizar.

### Tabla resumen de metricas

| Metrica | Pregunta que responde | Cuando priorizarla |
|---------|----------------------|--------------------|
| Accuracy | Que % de predicciones fueron correctas? | Clases balanceadas |
| Precision | De los que dije positivos, cuantos acerte? | Cuando FP es costoso (ej. spam) |
| Recall | De los positivos reales, cuantos detecte? | Cuando FN es costoso (ej. churn, enfermedad) |
| F1-Score | Balance entre Precision y Recall | Cuando ambos importan |

## 14. Actividad final (en equipos, 15-20 min)

Piensen en un problema de clasificacion binaria de su sector y respondan:

1. Cual es el problema? (Ejemplo: detectar si un pedido sera devuelto.)
2. Que variables usarian como predictoras? (Minimo 4 variables.)
3. Que es un Falso Positivo y un Falso Negativo en su problema?
4. Cual de los dos errores es mas costoso y por que?
5. Que metrica priorizarian: Precision o Recall?

Entregable: 1 slide o 1 pagina (texto) por equipo.